In [ ]:
!pip install -q -U \
transformers \
datasets \
accelerate \
peft \
trl \
bitsandbytes \
sentencepiece \
safetensors \
evaluate \
scikit-learn \
sacrebleu \
rouge-score \
bert-score \
sentence-transformers

In [ ]:
pip install cuda-toolkit==12.6.0

In [ ]:
pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import os
import sys
import json
import time
import gc

import torch
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [ ]:
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN cargado:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/TT2_colab")

DATA_DIR = DRIVE_ROOT / "data"
SFT_DIR = DATA_DIR / "sft_ready"
OUTPUT_DIR = DRIVE_ROOT / "outputs" / "lora_runs"

TEST_JSONL = SFT_DIR / "feina_repr30_test_sft.jsonl"

print("TEST_JSONL:", TEST_JSONL)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Existe TEST_JSONL:", TEST_JSONL.exists())
print("Existe OUTPUT_DIR:", OUTPUT_DIR.exists())

In [ ]:
MODEL_EVALS = [
    {
        "model_key": "llama3",
        "model_id": "meta-llama/Meta-Llama-3-8B-Instruct",
        "run_name": "lora_llama3_feina_repr30_v1",
    },
    {
        "model_key": "mistral",
        "model_id": "mistralai/Mistral-7B-Instruct-v0.2",
        "run_name": "lora_mistral_feina_repr30_v1",
    },
]

display(pd.DataFrame(MODEL_EVALS))

In [ ]:
data_files = {
    "test": str(TEST_JSONL),
}

dataset = load_dataset("json", data_files=data_files)
test_ds = dataset["test"]

print("test_ds:", test_ds)
print(test_ds[0].keys())
print(test_ds[0]["instruction"][:500])

assert test_ds.num_rows == 238, "Test no coincide con repr30"
print("OK test repr30 cargado correctamente")

In [ ]:
def recover_source_text_from_instruction(instruction: str) -> str:
    text = str(instruction)
    marker = "Texto:"
    end_marker = "Versión simplificada:"

    if marker in text:
        text = text.split(marker, 1)[1]
    if end_marker in text:
        text = text.split(end_marker, 1)[0]

    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = " ".join(text.split()).strip()
    return text

In [ ]:
compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

bnb_config

In [ ]:
def clean_lora_output(decoded_text: str) -> str:
    text = str(decoded_text).strip()
    marker = "Versión simplificada:"

    if marker in text:
        text = text.split(marker, 1)[-1].strip()

    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    text = " ".join(text.split()).strip()
    return text

In [ ]:
def load_lora_model(model_id: str, adapter_dir: Path):
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        use_fast=True,
        token=os.environ.get("HF_TOKEN"),
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print("pad_token:", tokenizer.pad_token)
    print("eos_token:", tokenizer.eos_token)

    t0 = time.perf_counter()
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        dtype=torch.float16,
        device_map="auto",
        token=os.environ.get("HF_TOKEN"),
    )

    model = PeftModel.from_pretrained(
        base_model,
        adapter_dir,
    )
    model.eval()

    t1 = time.perf_counter()
    print(f"Tiempo cargando modelo + adapter: {t1 - t0:.2f} s")

    return tokenizer, model

In [ ]:
def generate_simplification(model, tokenizer, instruction: str, max_input_length: int = 1024, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(
        instruction,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    cleaned = clean_lora_output(decoded)
    return cleaned

In [ ]:
if "/content" not in sys.path:
    sys.path.append("/content")

from src.evaluation.metrics import evaluate_dataframe, summarize_metrics

In [ ]:
def evaluate_one_lora_run(run_cfg: dict, test_ds):
    model_key = run_cfg["model_key"]
    model_id = run_cfg["model_id"]
    run_name = run_cfg["run_name"]

    run_dir = OUTPUT_DIR / run_name
    adapter_dir = run_dir / "final_adapter"
    eval_dir = run_dir / "evaluation"
    eval_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 90)
    print("Evaluando corrida")
    print("model_key :", model_key)
    print("model_id  :", model_id)
    print("run_name  :", run_name)
    print("adapter_dir:", adapter_dir)
    print("=" * 90)

    if not adapter_dir.exists():
        raise FileNotFoundError(f"No existe adapter_dir: {adapter_dir}")

    tokenizer, model = load_lora_model(model_id, adapter_dir)

    # prueba rápida
    for i in range(2):
        row = test_ds[i]
        pred = generate_simplification(model, tokenizer, row["instruction"], max_input_length=1024, max_new_tokens=256)
        print("-" * 90)
        print("ROW ID:", row["row_id"])
        print("REFERENCE:", row["output"][:250])
        print("PREDICTION:", pred[:250])

    records = []
    t0 = time.perf_counter()

    for i, row in enumerate(test_ds):
        pred = generate_simplification(
            model=model,
            tokenizer=tokenizer,
            instruction=row["instruction"],
            max_input_length=1024,
            max_new_tokens=256,
        )

        records.append({
            "row_id": row["row_id"],
            "instruction": row["instruction"],
            "generated_text": pred,
        })

        if (i + 1) % 25 == 0:
            print(f"Procesados: {i + 1}/{len(test_ds)}")

    t1 = time.perf_counter()
    generation_minutes = (t1 - t0) / 60
    print(f"Tiempo total de generación en test: {generation_minutes:.2f} min")

    lora_test_df = pd.DataFrame(records)
    lora_test_df["source_text"] = lora_test_df["instruction"].apply(recover_source_text_from_instruction)

    test_df_from_sft = pd.DataFrame(test_ds)
    test_df_from_sft = test_df_from_sft.rename(columns={"output": "reference_text"})

    lora_test_df = lora_test_df.merge(
        test_df_from_sft[["row_id", "reference_text"]],
        on="row_id",
        how="left",
    )

    print("Shape predicciones:", lora_test_df.shape)
    display(lora_test_df[["row_id", "source_text", "reference_text", "generated_text"]].head(3))

    pred_path = eval_dir / "lora_test_predictions.csv"
    lora_test_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
    print("Predicciones guardadas en:", pred_path)

    evaluated_lora_test_df = evaluate_dataframe(
        lora_test_df,
        source_col="source_text",
        pred_col="generated_text",
        ref_col="reference_text",
        compute_bertscore=True,
        compute_sbert=False,
    )

    display(evaluated_lora_test_df.head(3))

    lora_summary = summarize_metrics(
        evaluated_lora_test_df,
        group_cols=[],
    )
    display(lora_summary)

    evaluated_path = eval_dir / "lora_test_evaluated.csv"
    summary_path = eval_dir / "lora_test_summary.csv"

    evaluated_lora_test_df.to_csv(evaluated_path, index=False, encoding="utf-8-sig")
    lora_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

    print("Evaluated guardado en:", evaluated_path)
    print("Summary guardado en:", summary_path)

    main_metric_cols = [
        c for c in [
            "sari",
            "bertscore_f1",
            "bleu",
            "rouge1_f",
            "rouge2_f",
            "rougeL_f",
            "fernandez_huerta_pred",
            "inflesz_pred",
            "compression_ratio_eval",
            "exact_copy",
        ]
        if c in lora_summary.columns
    ]

    main_metrics = lora_summary[main_metric_cols].copy()
    display(main_metrics)

    result_row = {
        "run_name": run_name,
        "model_key": model_key,
        "model_id": model_id,
        "generation_minutes": generation_minutes,
    }

    for col in lora_summary.columns:
        result_row[col] = lora_summary.iloc[0][col]

    # liberar memoria
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result_row

In [ ]:
all_eval_results = []

for run_cfg in MODEL_EVALS:
    result_row = evaluate_one_lora_run(run_cfg, test_ds)
    all_eval_results.append(result_row)

eval_results_df = pd.DataFrame(all_eval_results)
display(eval_results_df)

In [ ]:
global_eval_path = OUTPUT_DIR / "lora_eval_comparison_repr30.csv"
eval_results_df.to_csv(global_eval_path, index=False, encoding="utf-8-sig")
print("Comparación global guardada en:", global_eval_path)

In [ ]:
print("Contenido de OUTPUT_DIR:")
for p in OUTPUT_DIR.iterdir():
    print("-", p.name)